# Week 1 — Foundations: Cleaning & Exploring the Applicant Dataset

**Name:** Khadija Anwer
**Program:** NextGen Learners — Data Science Internship
**Tools:** Python 3 + Pandas (Google Colab / Jupyter Notebook)

**Goal:** Take a messy `applicants.csv`, find every data-quality problem, fix each one **using code only** (no manual Excel edits), explain every decision in words, and export a clean `applicants_cleaned.csv`.

**Workflow:** 1 Setup → 2 Load data → 3 First look → 4 Missing values → 5 Duplicates → 6 Standardise text → 7 Fix data types → 8 Data Quality Summary → 9 Export → 10 Final checklist

## Step 1: Set Up the Environment

I am using a notebook instead of a plain script because I can run code in small cells and write my reasoning in markdown right next to the code. Pandas is the only library the cleaning itself needs.

In [1]:
import pandas as pd

## Step 2: Get (or Create) `applicants.csv`

My mentor did not provide a file, so I created a **mock dataset** with deliberately messy data (blank names/emails, extra whitespace, inconsistent casing, dates in two formats, dashes/spaces in phone numbers, exact duplicates and near-duplicates).

The function below builds it with a fixed random seed, so it produces the **same file every time**. It only runs if `applicants.csv` does not already exist, so if a real file is uploaded to Colab it is used instead.

Columns: `Applicant Name`, `Email`, `Phone`, `Domain Applied`, `University`, `Application Date`, `Status`.

In [2]:
import os
import random
from datetime import date, timedelta


def create_mock_applicants(path='applicants.csv', n_people=90, seed=42):
    """Create a deliberately messy applicant CSV (same seed -> same file)."""
    rng = random.Random(seed)

    first_names = ['Ali', 'Ayesha', 'Bilal', 'Fatima', 'Hassan', 'Hira', 'Imran', 'Iqra', 'Junaid', 'Kiran',
                   'Maryam', 'Usman', 'Noman', 'Sana', 'Zain', 'Areeba', 'Danish', 'Farah', 'Haris', 'Laiba',
                   'Mahnoor', 'Owais', 'Rabia', 'Saad', 'Tahir', 'Uzma', 'Waqar', 'Yusra', 'Zara', 'Adnan',
                   'Bushra', 'Faisal', 'Ghazal', 'Hamza', 'Irfan', 'Komal', 'Maaz', 'Nimra', 'Omer', 'Parveen']
    last_names = ['Khan', 'Ahmed', 'Siddiqui', 'Sheikh', 'Malik', 'Hussain', 'Raza', 'Qureshi', 'Ansari', 'Baig',
                  'Chaudhry', 'Farooqui', 'Iqbal', 'Javed', 'Memon', 'Naqvi', 'Rizvi', 'Shah', 'Tariq', 'Zaidi',
                  'Abbasi', 'Butt', 'Dar', 'Hashmi', 'Lodhi', 'Mirza', 'Niazi', 'Rehman', 'Soomro', 'Yousuf']
    mail_domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com']
    prefixes = ['0300', '0301', '0312', '0321', '0333', '0345']

    # messy spellings of the same real value
    domain_variants = {
        'Web Development': ['web dev', 'Web Development', 'WEB DEV', 'webdev', ' web dev ', 'Web development'],
        'Data Science': ['data science', 'Data Science', 'DATA SCIENCE', 'DS', 'data sci'],
        'Python Programming': ['python', 'Python Programming', 'PYTHON', 'python programming', 'Python'],
        'Graphic Design': ['graphic design', 'Graphic Design', 'GRAPHIC DESIGN', 'graphics design', 'graphic designing'],
        'Machine Learning': ['machine learning', 'Machine Learning', 'ML', 'ml', 'MACHINE LEARNING'],
    }
    university_variants = {
        'NED': ['NED University of Engineering and Technology', 'NED University', 'NED', 'ned university'],
        'KU': ['University of Karachi', 'Karachi University', 'KU', 'UoK'],
        'BBSUL': ['Benazir Bhutto Shaheed University Lyari', 'BBSUL', 'BBSU Lyari'],
        'FAST': ['FAST National University', 'FAST-NUCES', 'FAST NU', 'FAST'],
        'SSUET': ['Sir Syed University of Engineering and Technology', 'SSUET', 'Sir Syed University'],
        'IBA': ['Institute of Business Administration', 'IBA Karachi', 'IBA'],
        'IQRA': ['Iqra University', 'Iqra Univ', 'IQRA'],
    }
    status_variants = {
        'Under Review': ['Under Review', 'under review', 'UNDER REVIEW', 'Under review'],
        'Selected': ['Selected', 'selected', 'SELECTED', ' Selected '],
        'Rejected': ['Rejected', 'rejected', 'REJECTED', ' Rejected'],
    }

    start = date(2026, 3, 1)

    def fmt_date(d, iso):
        return d.strftime('%Y-%m-%d') if iso else d.strftime('%d/%m/%Y')

    # ---- base rows: unique people --------------------------------------------------
    combos = rng.sample([(f, l) for f in first_names for l in last_names], n_people)
    rows = []
    for f, l in combos:
        prefix = rng.choice(prefixes)
        digits = f'{rng.randint(0, 9999999):07d}'
        phone = rng.choice([f'{prefix}-{digits}', f'{prefix}-{digits}', f'{prefix} {digits}', f'{prefix}{digits}'])
        d = start + timedelta(days=rng.randint(0, 99))
        rows.append({
            'Applicant Name': f'{f} {l}',
            'Email': f'{f.lower()}.{l.lower()}{rng.randint(1, 99)}@{rng.choice(mail_domains)}',
            'Phone': phone,
            'Domain Applied': rng.choice(domain_variants[rng.choice(list(domain_variants))]),
            'University': rng.choice(university_variants[rng.choice(list(university_variants))]),
            'Application Date': fmt_date(d, rng.random() < 0.5),
            'Status': rng.choice(status_variants[rng.choices(list(status_variants), weights=[4, 3, 3])[0]]),
        })

    # ---- plant problems in separate (non-overlapping) rows ---------------------------
    idx = list(range(n_people))
    rng.shuffle(idx)
    miss_name, miss_email, miss_uni, miss_status = idx[0:3], idx[3:7], idx[7:12], idx[12:18]
    ws_name, bad_date, case_email = idx[18:26], idx[26:28], idx[28:38]
    exact_src, near_src = idx[38:41], idx[41:44]

    for i in miss_name:
        rows[i]['Applicant Name'] = ''
    for i in miss_email:
        rows[i]['Email'] = ''
    for i in miss_uni:
        rows[i]['University'] = ''
    for i in miss_status:
        rows[i]['Status'] = ''
    for i in ws_name:
        n = rows[i]['Applicant Name']
        rows[i]['Applicant Name'] = rng.choice([f' {n} ', f'{n}  ', f'  {n}'])
    for i in case_email:
        e = rows[i]['Email']
        rows[i]['Email'] = e.upper() if rng.random() < 0.5 else e.title()
    rows[bad_date[0]]['Application Date'] = 'TBD'
    rows[bad_date[1]]['Application Date'] = '31/02/2026'  # a date that does not exist

    # ---- duplicates -----------------------------------------------------------------
    extras = [dict(rows[i]) for i in exact_src]                      # exact copies
    for i in near_src:                                               # same person, re-submitted
        r = dict(rows[i])
        r['Applicant Name'] = f"  {r['Applicant Name'].lower()} "
        r['Email'] = r['Email'].upper()
        later = start + timedelta(days=rng.randint(100, 119))
        r['Application Date'] = fmt_date(later, rng.random() < 0.5)
        extras.append(r)
    rows += extras
    rng.shuffle(rows)

    pd.DataFrame(rows).to_csv(path, index=False)


if not os.path.exists('applicants.csv'):
    create_mock_applicants('applicants.csv')
    print('applicants.csv created (mock data).')
else:
    print('applicants.csv already exists - using it.')

# Phone is read as text on purpose: phone numbers start with 0 and must not become integers
df = pd.read_csv('applicants.csv', dtype={'Phone': str})
raw_df = df.copy()   # untouched copy -> used for the "before" snapshot at the end
print('Loaded:', df.shape)

applicants.csv created (mock data).
Loaded: (96, 7)


## Step 3: First Look — Understand Before You Clean

I never clean blindly. First I look at the shape, the data types, the missing values and the duplicates. This is my **"before" snapshot**.

In [3]:
print('Shape (rows, columns):', df.shape)
df.head()

Shape (rows, columns): (96, 7)


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Imran Baig,imran.baig85@outlook.com,0345 1422633,Web Development,IBA Karachi,2026-05-05,Under Review
1,Laiba Dar,laiba.dar29@hotmail.com,0321-0289293,DATA SCIENCE,IBA,2026-04-07,SELECTED
2,Imran Yousuf,imran.yousuf40@hotmail.com,0301-9047886,python programming,SSUET,2026-05-31,UNDER REVIEW
3,Zain Baig,NaN,03010044891,python,KU,27/04/2026,Rejected
4,Uzma Rehman,uzma.rehman7@yahoo.com,03459103209,ML,SSUET,22/04/2026,rejected


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Applicant Name    93 non-null     str  
 1   Email             92 non-null     str  
 2   Phone             96 non-null     str  
 3   Domain Applied    96 non-null     str  
 4   University        91 non-null     str  
 5   Application Date  96 non-null     str  
 6   Status            90 non-null     str  
dtypes: str(7)
memory usage: 5.4 KB


In [5]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Exact duplicate rows:', df.duplicated().sum())

Missing values per column:
Applicant Name      3
Email               4
Phone               0
Domain Applied      0
University          5
Application Date    0
Status              6
dtype: int64

Exact duplicate rows: 3


In [6]:
print('--- Domain Applied (raw spellings) ---')
print(df['Domain Applied'].value_counts().to_string())
print()
print('--- Status (raw) ---')
print(df['Status'].value_counts(dropna=False).to_string())
print()
print('--- A random sample of raw dates and phone numbers ---')
display(df[['Application Date', 'Phone']].sample(8, random_state=1))

--- Domain Applied (raw spellings) ---
Domain Applied
graphics design       9
Web development       7
graphic design        6
PYTHON                6
DS                    5
Python Programming    5
graphic designing     5
 web dev              5
DATA SCIENCE          4
python                4
ML                    4
Machine Learning      4
ml                    4
Data Science          4
Web Development       3
web dev               3
data sci              3
WEB DEV               3
python programming    2
GRAPHIC DESIGN        2
webdev                2
Graphic Design        2
data science          1
Python                1
machine learning      1
MACHINE LEARNING      1

--- Status (raw) ---
Status
Under review    12
 Selected       11
selected        10
 Rejected        9
REJECTED         8
Under Review     7
SELECTED         6
UNDER REVIEW     6
rejected         6
under review     6
NaN              6
Rejected         5
Selected         4

--- A random sample of raw dates and phone nu

,Application Date,Phone
40,06/04/2026,0312-9214376
31,2026-05-15,0300-9870182
46,06/06/2026,0345-4521644
59,11/05/2026,03017412769
78,07/04/2026,0345-2204724
74,2026-04-18,0312-2670896
65,2026-04-05,0300-8626108
44,24/05/2026,0301-1582524


**Observations (before cleaning):**

- The dataset has **96 rows and 7 columns**.
- Missing values: `Applicant Name` = 3, `Email` = 4, `University` = 5, `Status` = 6.
- There are **3 exact duplicate rows**. There may be more duplicates that `duplicated()` cannot see (same person, different formatting) — I check for those in Step 5.
- `Domain Applied` has **26 different spellings** for only 5 real tracks (casing, extra spaces, abbreviations like `DS` and `ML`).
- `Status` has inconsistent casing and extra spaces.
- `Application Date` is stored as text (`object`/`str`, not a date) and uses **two formats**: `DD/MM/YYYY` and `YYYY-MM-DD`.
- `Phone` mixes dashes, spaces and plain digits.

## Step 4: Handle Missing Values (Column by Column, With Justification)

There is no single correct rule. I decide per column, based on how important the column is.

| Column | Decision | Reason |
|---|---|---|
| Applicant Name | **Drop** the row | Without a name the application cannot be used for outreach |
| Email | **Drop** the row | Email is the follow-up channel and my duplicate check key |
| University | **Fill** with `Not Specified` | Not essential — the rest of the row is still valuable |
| Status | **Fill** with `Under Review` | No status most likely means "not processed yet" |

In [7]:
# Applicant Name: drop rows with no name
before = len(df)
df = df.dropna(subset=['Applicant Name'])
dropped_no_name = before - len(df)
print('Rows dropped (missing Applicant Name):', dropped_no_name)

Rows dropped (missing Applicant Name): 3


**Why:** I dropped 3 rows with a missing `Applicant Name`. A nameless application is not usable for outreach or reporting, so keeping it would only add noise.

In [8]:
# Email: drop rows with no email
before = len(df)
df = df.dropna(subset=['Email'])
dropped_no_email = before - len(df)
print('Rows dropped (missing Email):', dropped_no_email)

Rows dropped (missing Email): 4


**Why:** I dropped 4 rows with a missing `Email` instead of filling them with a placeholder like `not_provided@example.com`.

There is also a technical reason: in Step 5 I find duplicates **by email**. If I filled every blank with the *same* placeholder, `drop_duplicates(subset=['Email'])` would treat all those different people as duplicates of each other and silently delete all but one of them.

In [9]:
# University: fill with a placeholder (not critical)
filled_university = int(df['University'].isnull().sum())
df['University'] = df['University'].fillna('Not Specified')
print('Missing University filled with "Not Specified":', filled_university)

# Status: fill with the default for unprocessed applications
filled_status = int(df['Status'].isnull().sum())
df['Status'] = df['Status'].fillna('Under Review')
print('Missing Status filled with "Under Review":', filled_status)

Missing University filled with "Not Specified": 5
Missing Status filled with "Under Review": 6


**Why:** I kept the rows with a missing `University` (5 rows) and filled the gap with `Not Specified` because university is not essential to the application decision. Missing `Status` (6 rows) was filled with `Under Review` because an unrecorded status most likely means the application has not been processed yet.

In [10]:
print('Missing values after Step 4:')
print(df.isnull().sum())

Missing values after Step 4:
Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64


## Step 5: Remove Duplicate Applicants

Two kinds of duplicates exist:

1. **Exact duplicates** — every column identical (e.g. form double-submit). `drop_duplicates()` catches these.
2. **Near-duplicates** — the same person submits again but formatting differs (extra spaces, different capitalisation, a later date). `drop_duplicates()` cannot see these, so I compare on `Email`, which is a more reliable unique identifier than a name.

In [11]:
# 5a) exact duplicates
dropped_exact_dups = int(df.duplicated().sum())
print('Exact duplicate rows found:', dropped_exact_dups)
df = df.drop_duplicates()
print('Rows after removing exact duplicates:', len(df))

Exact duplicate rows found: 3
Rows after removing exact duplicates: 86


In [12]:
# 5b) near-duplicates: normalise the email first (strip spaces + lowercase) so "ALI@X.COM" == "ali@x.com"
df['Email'] = df['Email'].str.strip().str.lower()

near_dups = df[df.duplicated(subset=['Email'], keep=False)].sort_values('Email')
print('Rows that share an email with another row:', len(near_dups))
near_dups

Rows that share an email with another row: 6


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
25,maaz javed,maaz.javed97@yahoo.com,0345-4521644,graphic designing,IBA Karachi,10/06/2026,Under review
46,Maaz Javed,maaz.javed97@yahoo.com,0345-4521644,graphic designing,IBA Karachi,06/06/2026,Under review
37,owais rizvi,owais.rizvi57@gmail.com,0301 8427047,ml,NED University,12/06/2026,selected
42,Owais Rizvi,owais.rizvi57@gmail.com,0301 8427047,ml,NED University,2026-03-13,selected
36,Uzma Soomro,uzma.soomro99@gmail.com,03004107776,graphics design,IBA,2026-05-18,Selected
52,uzma soomro,uzma.soomro99@gmail.com,03004107776,graphics design,IBA,2026-06-28,Selected


In [13]:
before = len(df)
df = df.drop_duplicates(subset=['Email'], keep='first').reset_index(drop=True)
dropped_email_dups = before - len(df)
print('Near-duplicate rows removed (same Email):', dropped_email_dups)
print('Rows now:', len(df))

Near-duplicate rows removed (same Email): 3
Rows now: 83


**Explanation:** I removed **3 exact duplicate rows** and then **3 near-duplicates** found by matching email. In the table above you can see each pair has a differently formatted name and a different date, so only the email check could connect them. I checked by **email rather than full row or name** because a name can be written many ways (or shared by two different people), while an email address identifies one person. I kept the first submission (`keep='first'`).

## Step 6: Standardise Text Fields

The same real-world value can be written many ways, and a computer counts every spelling as a different category. I strip whitespace, fix casing, and map messy variants to one standard label.

In [14]:
# Names: strip invisible spaces and use Title Case
df['Applicant Name'] = df['Applicant Name'].str.strip().str.title()
df['Applicant Name'].head()

0      Imran Baig
1       Laiba Dar
2    Imran Yousuf
3     Uzma Rehman
4    Hassan Niazi
Name: Applicant Name, dtype: str

In [15]:
# Domain Applied: see every variant (after strip + lowercase) before building the mapping
sorted(df['Domain Applied'].str.strip().str.lower().unique())

['data sci', 'data science', 'ds', 'graphic design', 'graphic designing', 'graphics design', 'machine learning', 'ml', 'python', 'python programming', 'web dev', 'web development', 'webdev']

In [16]:
domain_mapping = {
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',
    'data science': 'Data Science',
    'ds': 'Data Science',
    'data sci': 'Data Science',
    'python': 'Python Programming',
    'python programming': 'Python Programming',
    'graphic design': 'Graphic Design',
    'graphics design': 'Graphic Design',
    'graphic designing': 'Graphic Design',
    'machine learning': 'Machine Learning',
    'ml': 'Machine Learning',
}

df['Domain Applied'] = df['Domain Applied'].str.strip().str.lower().map(domain_mapping)

print('Unmapped (NaN) domains:', df['Domain Applied'].isnull().sum())
print(df['Domain Applied'].unique())
print(df['Domain Applied'].value_counts().to_string())

Unmapped (NaN) domains: 0
<StringArray>
['Web Development', 'Data Science', 'Python Programming', 'Machine Learning', 'Graphic Design']
Length: 5, dtype: str
Domain Applied
Web Development       21
Graphic Design        19
Python Programming    16
Data Science          15
Machine Learning      12


**Explanation:** I first lowercased and stripped the column and listed every variant, then built a dictionary that maps each one to a standard label. After mapping I re-checked with `.unique()`: there are **no `NaN` values** (so no variant was missed) and exactly **5 clean domains**.

In [17]:
# Status: strip + Title Case -> "selected", "SELECTED", " Selected " all become "Selected"
df['Status'] = df['Status'].str.strip().str.title()
print(df['Status'].value_counts().to_string())

Status
Under Review    32
Selected        28
Rejected        23


In [18]:
# University: same idea, abbreviations -> full name
university_variants = {
    'NED University of Engineering and Technology': ['ned university of engineering and technology', 'ned university', 'ned'],
    'University of Karachi': ['university of karachi', 'karachi university', 'ku', 'uok'],
    'Benazir Bhutto Shaheed University Lyari': ['benazir bhutto shaheed university lyari', 'bbsul', 'bbsu lyari'],
    'FAST National University': ['fast national university', 'fast-nuces', 'fast nu', 'fast'],
    'Sir Syed University of Engineering and Technology': ['sir syed university of engineering and technology', 'ssuet', 'sir syed university'],
    'Institute of Business Administration': ['institute of business administration', 'iba karachi', 'iba'],
    'Iqra University': ['iqra university', 'iqra univ', 'iqra'],
    'Not Specified': ['not specified'],
}
# invert it: {'ned': 'NED University of ...', ...}
university_mapping = {v: canon for canon, variants in university_variants.items() for v in variants}

print('Distinct University spellings before:', df['University'].nunique())
df['University'] = df['University'].str.strip().str.lower().map(university_mapping)
print('Unmapped (NaN) universities:', df['University'].isnull().sum())
print('Distinct University values after:', df['University'].nunique())

Distinct University spellings before: 24
Unmapped (NaN) universities: 0
Distinct University values after: 8


## Step 7: Fix Data Types (Dates and Phone Numbers)

A date stored as text looks right to a human, but code cannot sort, filter or calculate with it until it becomes a real date type.

**Important detail:** the file has two formats: `12/05/2026` (day first) **and** `2026-05-12`. In recent Pandas versions, `to_datetime` guesses one format from the first value and turns every non-matching value into `NaT`. Using `dayfirst=True` on everything is also unsafe: it can flip day and month of the `YYYY-MM-DD` values (`2026-03-02` would become 3 February). So I parse each format **explicitly**, then combine the two results. `errors='coerce'` turns anything that matches neither format into `NaT` instead of crashing the notebook.

In [19]:
raw_dates = df['Application Date'].copy()
print('Missing dates before conversion:', raw_dates.isnull().sum())

# parse each known format explicitly, then combine (a value that fits neither format stays NaT)
dates_iso = pd.to_datetime(df['Application Date'], format='%Y-%m-%d', errors='coerce')   # 2026-05-12
dates_dmy = pd.to_datetime(df['Application Date'], format='%d/%m/%Y', errors='coerce')   # 12/05/2026
df['Application Date'] = dates_iso.fillna(dates_dmy)

n_bad_dates = int(df['Application Date'].isnull().sum())
print('Dates that could not be understood (now NaT):', n_bad_dates)

bad_mask = df['Application Date'].isnull()
pd.DataFrame({'Applicant Name': df.loc[bad_mask, 'Applicant Name'], 'Original value': raw_dates[bad_mask]})

Missing dates before conversion: 0
Dates that could not be understood (now NaT): 2


,Applicant Name,Original value
16,Fatima Sheikh,TBD
40,Usman Malik,31/02/2026


**Decision:** 2 dates were unreadable (`TBD` and `31/02/2026`, a date that does not exist). I **kept** these applicants and left the date as `NaT`: the person, email and domain are valid, and I should not invent a date. They are flagged above so a mentor can follow up with the applicant.

In [20]:
# sanity check: all parsed dates should fall inside the application window (Mar-Jun 2026)
valid = df['Application Date'].dropna()
print('Earliest:', valid.min().date(), '| Latest:', valid.max().date())
assert valid.between('2026-03-01', '2026-06-30').all(), 'Some dates fall outside the window - day/month may be swapped'

Earliest: 2026-03-02 | Latest: 2026-06-12


In [21]:
# Phone: remove dashes and spaces; keep as TEXT so the leading 0 is not lost
df['Phone'] = df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False)

# validity check: every number should be 11 digits starting with 03
print('All phones are 11 digits starting with 03:', df['Phone'].str.fullmatch(r'03\d{9}').all())
df['Phone'].head()

All phones are 11 digits starting with 03: True


0    03451422633
1    03210289293
2    03019047886
3    03459103209
4    03001000560
Name: Phone, dtype: str

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    83 non-null     str           
 1   Email             83 non-null     str           
 2   Phone             83 non-null     str           
 3   Domain Applied    83 non-null     str           
 4   University        83 non-null     str           
 5   Application Date  81 non-null     datetime64[us]
 6   Status            83 non-null     str           
dtypes: datetime64[us](1), str(6)
memory usage: 4.7 KB


`Application Date` is now a real `datetime64` column and `Phone` is consistent text.

## Step 8: Data Quality Summary

First a side-by-side "before vs after" table computed directly from the data, then the written summary.

In [23]:
report = pd.DataFrame(
    {
        'Before (raw)': [
            raw_df.shape[0], raw_df.shape[1],
            raw_df['Applicant Name'].isnull().sum(), raw_df['Email'].isnull().sum(),
            raw_df['University'].isnull().sum(), raw_df['Status'].isnull().sum(),
            raw_df.duplicated().sum(),
            raw_df['Email'].dropna().str.strip().str.lower().duplicated().sum(),
            raw_df['Domain Applied'].nunique(), raw_df['University'].nunique(), raw_df['Status'].nunique(),
            str(raw_df['Application Date'].dtype),
        ],
        'After (clean)': [
            df.shape[0], df.shape[1],
            df['Applicant Name'].isnull().sum(), df['Email'].isnull().sum(),
            df['University'].isnull().sum(), df['Status'].isnull().sum(),
            df.duplicated().sum(),
            df['Email'].duplicated().sum(),
            df['Domain Applied'].nunique(), df['University'].nunique(), df['Status'].nunique(),
            str(df['Application Date'].dtype),
        ],
    },
    index=['Rows', 'Columns', 'Missing Applicant Name', 'Missing Email', 'Missing University', 'Missing Status',
           'Exact duplicate rows', 'Duplicate emails', 'Distinct Domain values', 'Distinct University values',
           'Distinct Status values', 'Application Date dtype'],
)
report

,Before (raw),After (clean)
Rows,96,83
Columns,7,7
Missing Applicant Name,3,0
Missing Email,4,0
Missing University,5,0
Missing Status,6,0
Exact duplicate rows,3,0
Duplicate emails,6,0
Distinct Domain values,26,5
Distinct University values,24,8


### Data Quality Summary

**Starting dataset:** 96 rows, 7 columns

**Issues found:**
- Missing values: 3 in Applicant Name, 4 in Email, 5 in University, 6 in Status
- 3 exact duplicate rows, plus 3 additional duplicate applications identified by matching Email
- Domain Applied had 26 different spellings (casing, spaces, abbreviations) for only 5 real domains
- University had 24 different spellings for 7 real universities
- Status had inconsistent casing and extra whitespace
- Application Date was stored as text in two formats (`DD/MM/YYYY` and `YYYY-MM-DD`); 2 values were unreadable
- Phone numbers contained inconsistent dash/space formatting; names had stray whitespace; emails had mixed capitalisation

**Actions taken:**
- Dropped 3 rows with missing Applicant Name (unusable without identification)
- Dropped 4 rows with missing Email (needed for follow-up and duplicate detection)
- Filled 5 missing University values with "Not Specified" (non-critical field)
- Filled 6 missing Status values with "Under Review" (default for unprocessed applications)
- Removed 6 duplicate rows in total (3 exact, 3 by matching Email)
- Stripped whitespace, applied Title Case to names, lowercased emails
- Standardised Domain Applied into 5 categories, University into 7 (+ "Not Specified"), Status into 3
- Converted Application Date to `datetime64` (each format parsed explicitly with `pd.to_datetime(..., errors='coerce')`); kept 2 unreadable dates as `NaT`
- Cleaned Phone numbers (dashes and spaces removed, kept as text)

**Final dataset:** 83 rows, 7 columns, no missing values in Applicant Name, Email, University or Status, no exact or near duplicates, consistent formatting throughout.

## Step 9: Export the Cleaned Dataset

`index=False` stops Pandas from adding an extra unnamed column of row numbers.

In [24]:
df.to_csv('applicants_cleaned.csv', index=False)

# confirm the file saved correctly by reading it back
check = pd.read_csv('applicants_cleaned.csv', dtype={'Phone': str})
print('Saved applicants_cleaned.csv ->', check.shape)
check.head()

Saved applicants_cleaned.csv -> (83, 7)


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Imran Baig,imran.baig85@outlook.com,03451422633,Web Development,Institute of Business Administration,2026-05-05,Under Review
1,Laiba Dar,laiba.dar29@hotmail.com,03210289293,Data Science,Institute of Business Administration,2026-04-07,Selected
2,Imran Yousuf,imran.yousuf40@hotmail.com,03019047886,Python Programming,Sir Syed University of Engineering and Technology,2026-05-31,Under Review
3,Uzma Rehman,uzma.rehman7@yahoo.com,03459103209,Machine Learning,Sir Syed University of Engineering and Technology,2026-04-22,Rejected
4,Hassan Niazi,hassan.niazi77@outlook.com,03001000560,Graphic Design,FAST National University,2026-04-09,Rejected


## Step 10: Final Checklist

In [25]:
checks = {
    'No missing Applicant Name / Email': df[['Applicant Name', 'Email']].isnull().sum().sum() == 0,
    'No missing University / Status': df[['University', 'Status']].isnull().sum().sum() == 0,
    'No exact duplicate rows': df.duplicated().sum() == 0,
    'No duplicate emails (near-duplicates)': df['Email'].duplicated().sum() == 0,
    'Domain Applied has exactly 5 clean values': df['Domain Applied'].nunique() == 5,
    'Status has exactly 3 clean values': df['Status'].nunique() == 3,
    'Application Date is datetime64': pd.api.types.is_datetime64_any_dtype(df['Application Date']),
    'Phones are 11 digits starting with 03': df['Phone'].str.fullmatch(r'03\d{9}').all(),
    'applicants_cleaned.csv exported': os.path.exists('applicants_cleaned.csv'),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values()), 'Some checks failed'

PASS - No missing Applicant Name / Email
PASS - No missing University / Status
PASS - No exact duplicate rows
PASS - No duplicate emails (near-duplicates)
PASS - Domain Applied has exactly 5 clean values
PASS - Status has exactly 3 clean values
PASS - Application Date is datetime64
PASS - Phones are 11 digits starting with 03
PASS - applicants_cleaned.csv exported


**Reproducibility:** every fix above is code — no manual editing in Excel. Running *Runtime → Run all* from the top recreates `applicants.csv` (or loads the uploaded one) and regenerates `applicants_cleaned.csv`.

**Pandas functions used:** `isnull().sum()`, `duplicated()`, `drop_duplicates()`, `dropna()`, `fillna()`, `str.strip()`, `str.lower()`, `str.title()`, `map()`, `pd.to_datetime()`, `str.replace()`.